In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore')


In [ ]:

# ================================================================
# STEP 1: PATHS AND SETTINGS
# ================================================================

TRAIN_PATH = "/kaggle/input/datasets/dhruvmak/face-mask-detection/dataset"
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 10


In [ ]:

# ================================================================
# STEP 2: DATA AUGMENTATION AND LOADING
# ================================================================

train_datagen = ImageDataGenerator(
    rescale            = 1./255,
    rotation_range     = 20,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    validation_split   = 0.2
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'training'
)

val_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'validation'
)

print("Classes Found:", train_generator.class_indices)
print(f"Training Samples:   {train_generator.samples}")
print(f"Validation Samples: {val_generator.samples}")


In [ ]:

# ================================================================
# STEP 3: SHOW 2 SAMPLE IMAGES
# ================================================================

images, labels = next(train_generator)
class_names    = {v: k for k, v in train_generator.class_indices.items()}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i in range(2):
    axes[i].imshow(images[i])
    axes[i].set_title(f"Label: {class_names[int(labels[i])]}", fontsize=14)
    axes[i].axis('off')
plt.suptitle("Sample Images from Dataset", fontsize=16)
plt.tight_layout()
plt.show()

# ================================================================
# STEP 4: LOAD VGG16 (Transfer Learning)
# ================================================================

base_model = VGG16(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (224, 224, 3)
)

base_model.trainable = False

print(f"VGG16 Loaded  - Total Layers: {len(base_model.layers)}")

# ================================================================
# STEP 5: BUILD FULL MODEL
# ================================================================

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=0.0001),
    loss      = 'binary_crossentropy',
    metrics   = ['accuracy']
)

model.summary()


In [ ]:

# ================================================================
# STEP 6: TRAIN THE MODEL
# ================================================================

history = model.fit(
    train_generator,
    epochs          = EPOCHS,
    validation_data = val_generator
)


In [ ]:

# ================================================================
# STEP 7: PLOT TRAINING RESULTS
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss',   linewidth=2)
axes[0].set_title('Loss Over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   linewidth=2)
axes[1].set_title('Accuracy Over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

# ================================================================
# STEP 8: PLOT PREDICTIONS ON VALIDATION IMAGES
# ================================================================

val_images, val_labels = next(val_generator)

predictions = model.predict(val_images)
pred_labels = (predictions > 0.5).astype(int).flatten()

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes      = axes.flatten()

for i in range(12):
    axes[i].imshow(val_images[i])

    true_label = class_names[int(val_labels[i])]
    pred_label = class_names[pred_labels[i]]
    confidence = predictions[i][0] if pred_labels[i] == 1 else 1 - predictions[i][0]

    color = 'green' if true_label == pred_label else 'red'

    axes[i].set_title(
        f"True: {true_label}\nPred: {pred_label}\nConf: {confidence*100:.1f}%",
        color    = color,
        fontsize = 10
    )
    axes[i].axis('off')

plt.suptitle("Predictions  (Green = Correct  |  Red = Wrong)", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:

# ================================================================
# STEP 9: FINAL ACCURACY
# ================================================================

val_loss, val_acc = model.evaluate(val_generator)
print(f"\nFinal Validation Loss:     {val_loss:.4f}")
print(f"Final Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_single_image(img_path, model, class_names):
    # 1. Load and resize the image
    img = image.load_img(img_path, target_size=(224, 224))
    
    # 2. Convert to array and normalize (1./255)
    img_array = image.img_to_array(img) / 255.0
    
    # 3. Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # 4. Make prediction
    prediction = model.predict(img_batch)
    
    # 5. Interpret the result
    # For binary classification: 0 is usually 'with_mask' and 1 is 'without_mask' 
    # (depending on your folder names alphabetical order)
    prob = prediction[0][0]
    pred_label = 1 if prob > 0.5 else 0
    confidence = prob if pred_label == 1 else 1 - prob
    
    class_label = class_names[pred_label]
    
    print(f"Prediction: {class_label}")
    print(f"Confidence: {confidence * 100:.2f}%")
    
    # Optional: Display the image
    plt.imshow(img)
    plt.title(f"{class_label} ({confidence * 100:.1f}%)")
    plt.axis('off')
    plt.show()
# Example image path: /kaggle/input/datasets/dhruvmak/face-mask-detection/dataset/without_mask/image_1.png
# ==========================================
# TEST IT ON YOUR IMAGE
# ==========================================
# Replace 'your_test_image.jpg' with your actual file path
my_image_path = "/kaggle/input/datasets/huzifaulhaq/without-mask/images.jpg" 
predict_single_image(my_image_path, model, class_names)


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

def predict_single_image(img_path, model, class_names):
    # 1. Load and resize the image
    img = image.load_img(img_path, target_size=(224, 224))
    
    # 2. Convert to array and normalize (1./255)
    img_array = image.img_to_array(img) / 255.0
    
    # 3. Add batch dimension: (224, 224, 3) -> (1, 224, 224, 3)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # 4. Make prediction
    prediction = model.predict(img_batch)
    
    # 5. Interpret the result
    # For binary classification: 0 is usually 'with_mask' and 1 is 'without_mask' 
    # (depending on your folder names alphabetical order)
    prob = prediction[0][0]
    pred_label = 1 if prob > 0.5 else 0
    confidence = prob if pred_label == 1 else 1 - prob
    
    class_label = class_names[pred_label]
    
    print(f"Prediction: {class_label}")
    print(f"Confidence: {confidence * 100:.2f}%")
    
    # Optional: Display the image
    plt.imshow(img)
    plt.title(f"{class_label} ({confidence * 100:.1f}%)")
    plt.axis('off')
    plt.show()
# ==========================================
# TEST IT ON YOUR IMAGE
# ==========================================
# Replace 'your_test_image.jpg' with your actual file path
my_image_path = "/kaggle/input/datasets/dhruvmak/face-mask-detection/dataset/without_mask/image_1.png" 
predict_single_image(my_image_path, model, class_names)

# Birds Species Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import os
import warnings
warnings.filterwarnings('ignore')


In [ ]:

# ================================================================
# STEP 1: PATHS AND SETTINGS
# ================================================================

TRAIN_PATH = "/kaggle/input/datasets/akash2907/bird-species-classification/train_data/train_data"
TEST_PATH  = "/kaggle/input/datasets/akash2907/bird-species-classification/test_data/test_data"
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 10


In [ ]:

# ================================================================
# STEP 2: DATA AUGMENTATION AND LOADING (CATEGORICAL)
# ================================================================

# Augmentation for training
train_datagen = ImageDataGenerator(
    rescale            = 1./255,
    rotation_range     = 30,
    width_shift_range  = 0.2,
    height_shift_range = 0.2,
    shear_range        = 0.2,
    zoom_range         = 0.2,
    horizontal_flip    = True,
    fill_mode          = 'nearest'
)

# Only rescaling for testing/validation
test_datagen = ImageDataGenerator(rescale = 1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical' # Changed to categorical for many species
)

val_generator = test_datagen.flow_from_directory(
    TEST_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical'
)

# Get dynamic number of classes and class names
NUM_CLASSES = len(train_generator.class_indices)
class_names = {v: k for k, v in train_generator.class_indices.items()}

print(f"\nTotal Bird Species Detected: {NUM_CLASSES}")
print(f"Training Samples: {train_generator.samples}")

# ================================================================
# STEP 3: LOAD VGG16 (Transfer Learning)
# ================================================================

base_model = VGG16(
    weights     = 'imagenet',
    include_top = False,
    input_shape = (224, 224, 3)
)

base_model.trainable = False # Freeze VGG16 layers


In [ ]:

# ================================================================
# STEP 4: BUILD MULTI-CLASS MODEL
# ================================================================

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax') # Softmax gives probabilities for all classes
])

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=0.0001),
    loss      = 'categorical_crossentropy', # Required for multi-class
    metrics   = ['accuracy']
)

model.summary()


In [ ]:

# ================================================================
# STEP 5: TRAIN THE MODEL
# ================================================================

history = model.fit(
    train_generator,
    epochs          = EPOCHS,
    validation_data = val_generator
)


In [ ]:

# ================================================================
# STEP 6: PLOT TRAINING RESULTS
# ================================================================

def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss Plot
    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss History')
    axes[0].legend()
    
    # Accuracy Plot
    axes[1].plot(history.history['accuracy'], label='Train Acc')
    axes[1].plot(history.history['val_accuracy'], label='Val Acc')
    axes[1].set_title('Accuracy History')
    axes[1].legend()
    
    plt.show()

plot_history(history)


In [ ]:

# ================================================================
# STEP 7: PREDICTION FUNCTION FOR SINGLE IMAGE
# ================================================================

def predict_bird_species(img_path, model, class_names):
    # Load and preprocess image
    img = load_img(img_path, target_size=(224, 224))
    img_array = img_to_array(img) / 255.0
    img_batch = np.expand_dims(img_array, axis=0) # Add batch dimension
    
    # Make prediction
    preds = model.predict(img_batch)
    top_idx = np.argmax(preds[0]) # Index of highest probability
    confidence = preds[0][top_idx]
    
    result = class_names[top_idx]
    
    print(f"\n--- Prediction Result ---")
    print(f"Species: {result}")
    print(f"Confidence: {confidence*100:.2f}%")
    
    plt.imshow(img)
    plt.title(f"Pred: {result} ({confidence*100:.1f}%)")
    plt.axis('off')
    plt.show()

# To use it, simply provide an image path:
# predict_bird_species("path/to/test/image.jpg", model, class_names)